In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df1 = pd.read_csv("datasets/dataset-tickets-multi-lang-4-20k.csv")
df2 = pd.read_csv("datasets/aa_dataset-tickets-multi-lang-5-2-50-version.csv")
df3 = pd.read_csv("datasets/dataset-tickets-multi-lang3-4k.csv")

fdf = pd.concat([df1, df2, df3], ignore_index=True)


fdf = fdf.dropna(subset=['body']).copy()
fdf['text'] = fdf['subject'].fillna('') + ' ' + fdf['body']
fdf = fdf.drop_duplicates(subset='text').reset_index(drop=True)
df = fdf[['text', 'type', 'priority']].copy()

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['type'], random_state=42)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(train_df.shape, test_df.shape)   # (20109, 3) (5028, 3) 

(20109, 3) (5028, 3)


In [5]:
import time
from sklearn.base import clone
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
assert X_train.shape[0] == len(train_df) and X_test.shape[0] == len(test_df)

models = {
    'LinearSVC': LinearSVC(class_weight='balanced', max_iter=5000),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=2, random_state=42),
}

for name, m in models.items():
    for target in ['type', 'priority']:
        start = time.time()
        clf = clone(m)
        clf.fit(X_train, train_df[target])
        pred = clf.predict(X_test)
        print("=====", name, "|", target, "|", round(time.time() - start), "sec")
        print("accuracy:", round(accuracy_score(test_df[target], pred), 3),
              "| macro-F1:", round(f1_score(test_df[target], pred, average='macro'), 3))
        print(classification_report(test_df[target], pred))

===== LinearSVC | type | 12 sec
accuracy: 0.783 | macro-F1: 0.784
              precision    recall  f1-score   support

      Change       0.92      0.92      0.92       532
    Incident       0.74      0.78      0.76      2005
     Problem       0.52      0.46      0.49      1040
     Request       0.96      0.96      0.96      1451

    accuracy                           0.78      5028
   macro avg       0.79      0.78      0.78      5028
weighted avg       0.78      0.78      0.78      5028

===== LinearSVC | priority | 24 sec
accuracy: 0.478 | macro-F1: 0.457
              precision    recall  f1-score   support

        high       0.54      0.55      0.54      1939
         low       0.34      0.36      0.35      1019
      medium       0.50      0.47      0.48      2070

    accuracy                           0.48      5028
   macro avg       0.46      0.46      0.46      5028
weighted avg       0.48      0.48      0.48      5028

===== RandomForest | type | 87 sec
accuracy: 0.8

In [7]:
sims, nn = [], []
for i in range(0, len(X_test), 500):
    chunk = X_test[i:i+500] @ X_train.T      
    sims.append(chunk.max(axis=1))
    nn.append(chunk.argmax(axis=1))
max_sim = np.concatenate(sims)
nn = np.concatenate(nn)

print("similarity percentiles (5,25,50,75,95):", np.percentile(max_sim, [5, 25, 50, 75, 95]).round(3))
print("test rows with sim > 0.95:", round((max_sim > 0.95).mean(), 3))


print("1-NN type acc:", round((train_df['type'].values[nn] == test_df['type'].values).mean(), 3))
print("1-NN priority acc:", round((train_df['priority'].values[nn] == test_df['priority'].values).mean(), 3))

similarity percentiles (5,25,50,75,95): [0.857 0.929 0.966 0.985 0.996]
test rows with sim > 0.95: 0.625
1-NN type acc: 0.894
1-NN priority acc: 0.81


In [8]:
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

X_all = np.vstack([X_train, X_test])
all_df = pd.concat([train_df, test_df], ignore_index=True)

def make_groups(X, thr):
    rows, cols = [], []
    for i in range(0, len(X), 500):
        sim = X[i:i+500] @ X.T
        r, c = np.where(sim > thr)
        rows.append(r + i)
        cols.append(c)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(X), len(X)))
    return connected_components(g, directed=False)[1]

groups = make_groups(X_all, 0.95)
sizes = np.bincount(groups)
print("groups:", len(sizes), "| largest group:", sizes.max(), "| singletons:", (sizes == 1).sum())

groups: 13564 | largest group: 499 | singletons: 7766


In [9]:
for thr in [0.95, 0.93]:
    g = make_groups(X_all, thr)
    s = np.bincount(g)
    print(thr, "| groups:", len(s), "| largest:", s.max(), "| singletons:", (s == 1).sum())

0.95 | groups: 13564 | largest: 499 | singletons: 7766
0.93 | groups: 9975 | largest: 1042 | singletons: 5063


In [10]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, te_idx = next(sgkf.split(X_all, all_df['type'], groups))

X_train2, X_test2 = X_all[tr_idx], X_all[te_idx]
train2 = all_df.iloc[tr_idx].reset_index(drop=True)
test2 = all_df.iloc[te_idx].reset_index(drop=True)

print(len(train2), len(test2))
print(test2['type'].value_counts(normalize=True).round(3))
print(test2['priority'].value_counts(normalize=True).round(3))

20110 5027
type
Incident    0.399
Request     0.289
Problem     0.207
Change      0.106
Name: proportion, dtype: float64
priority
medium    0.424
high      0.382
low       0.194
Name: proportion, dtype: float64


In [11]:
sims, nn = [], []
for i in range(0, len(X_test2), 500):
    chunk = X_test2[i:i+500] @ X_train2.T
    sims.append(chunk.max(axis=1))
    nn.append(chunk.argmax(axis=1))
mx, nn = np.concatenate(sims), np.concatenate(nn)

print("max similarity test->train:", mx.max().round(3))
print("1-NN type acc:", round((train2['type'].values[nn] == test2['type'].values).mean(), 3))
print("1-NN priority acc:", round((train2['priority'].values[nn] == test2['priority'].values).mean(), 3))

max similarity test->train: 0.95
1-NN type acc: 0.739
1-NN priority acc: 0.461


In [12]:
np.save('tr_idx.npy', tr_idx)   # split reproducible rahe
np.save('te_idx.npy', te_idx)

train_df, test_df = train2, test2
X_train, X_test = X_train2, X_test2

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

models = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'LR': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'LinearSVC': LinearSVC(class_weight='balanced', max_iter=5000),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=2, random_state=42),
}

results = []
for name, m in models.items():
    for target in ['type', 'priority']:
        clf = clone(m)
        clf.fit(X_train, train_df[target])
        pred = clf.predict(X_test)
        results.append({
            'model': name, 'target': target,
            'accuracy': round(accuracy_score(test_df[target], pred), 3),
            'macro_F1': round(f1_score(test_df[target], pred, average='macro'), 3)
        })
        print(results[-1])

pd.DataFrame(results).pivot(index='model', columns='target', values=['accuracy', 'macro_F1'])

{'model': 'Baseline', 'target': 'type', 'accuracy': 0.399, 'macro_F1': 0.143}
{'model': 'Baseline', 'target': 'priority', 'accuracy': 0.424, 'macro_F1': 0.198}
{'model': 'LR', 'target': 'type', 'accuracy': 0.745, 'macro_F1': 0.764}
{'model': 'LR', 'target': 'priority', 'accuracy': 0.416, 'macro_F1': 0.408}
{'model': 'LinearSVC', 'target': 'type', 'accuracy': 0.774, 'macro_F1': 0.773}
{'model': 'LinearSVC', 'target': 'priority', 'accuracy': 0.434, 'macro_F1': 0.408}
{'model': 'RandomForest', 'target': 'type', 'accuracy': 0.765, 'macro_F1': 0.739}
{'model': 'RandomForest', 'target': 'priority', 'accuracy': 0.469, 'macro_F1': 0.393}


accuracy        macro_F1       
target       priority   type priority   type
model                                       
Baseline        0.424  0.399    0.198  0.143
LR              0.416  0.745    0.408  0.764
LinearSVC       0.434  0.774    0.408  0.773
RandomForest    0.469  0.765    0.393  0.739

In [13]:
dummy = DummyClassifier(strategy='stratified', random_state=42)
dummy.fit(X_train, train_df['priority'])
print(f1_score(test_df['priority'], dummy.predict(X_test), average='macro'))

0.3346410415398117


In [14]:
import joblib
from sklearn.linear_model import LogisticRegression

final = {}
for target in ['type', 'priority']:
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, train_df[target])
    final[target] = clf
    joblib.dump(clf, f'clf_{target}.joblib')

In [15]:
for target in ['type', 'priority']:
    proba = final[target].predict_proba(X_test)
    conf = proba.max(axis=1)
    pred = final[target].classes_[proba.argmax(axis=1)]
    correct = (pred == test_df[target].values)
    print("=====", target)
    for thr in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8]:
        keep = conf >= thr
        acc = correct[keep].mean() if keep.any() else float('nan')
        print(f"threshold {thr}: auto-handled {keep.mean():.0%} | accuracy on those {acc:.3f}")

===== type
threshold 0.0: auto-handled 100% | accuracy on those 0.745
threshold 0.4: auto-handled 99% | accuracy on those 0.749
threshold 0.5: auto-handled 95% | accuracy on those 0.763
threshold 0.6: auto-handled 67% | accuracy on those 0.841
threshold 0.7: auto-handled 47% | accuracy on those 0.930
threshold 0.8: auto-handled 35% | accuracy on those 0.987
===== priority
threshold 0.0: auto-handled 100% | accuracy on those 0.416
threshold 0.4: auto-handled 73% | accuracy on those 0.428
threshold 0.5: auto-handled 27% | accuracy on those 0.521
threshold 0.6: auto-handled 11% | accuracy on those 0.662
threshold 0.7: auto-handled 4% | accuracy on those 0.816
threshold 0.8: auto-handled 1% | accuracy on those 0.902


In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
model.max_seq_length = 256

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
import os
import pandas as pd
from datetime import datetime

THRESHOLDS = {'type': 0.7, 'priority': 0.0}   # priority 0.0 = kabhi review trigger nahi karega  
LOG_FILE = 'low_confidence_log.csv'

def triage(texts):
    emb = model.encode(texts, normalize_embeddings=True)
    out = pd.DataFrame({'text': texts})
    flag = pd.Series(False, index=out.index)

    for target in ['type', 'priority']:
        proba = final[target].predict_proba(emb)
        out[target] = final[target].classes_[proba.argmax(axis=1)]
        out[target + '_conf'] = proba.max(axis=1).round(3)
        flag |= out[target + '_conf'] < THRESHOLDS[target]

    out['priority_reliable'] = out['priority_conf'] >= 0.6
    out['needs_review'] = flag
    out['time'] = datetime.now().isoformat(timespec='seconds')

    low = out[out['needs_review']]
    if len(low) > 0:
        low.to_csv(LOG_FILE, mode='a', header=not os.path.exists(LOG_FILE), index=False)
    return out

In [25]:
sample_rows = test_df.sample(100, random_state=1).reset_index(drop=True)

sample = sample_rows["text"].tolist()

res = triage(sample)

res["actual_type"] = sample_rows["type"]
res["actual_priority"] = sample_rows["priority"]

from IPython.display import display

display(res[
    [
        "type",
        "actual_type",
        "type_conf",
        "priority",
        "actual_priority",
        "priority_conf",
        "needs_review"
    ]
])

print("flagged:", res["needs_review"].sum(), "of", len(res))
print(pd.read_csv(LOG_FILE).shape)

,type,actual_type,type_conf,priority,actual_priority,priority_conf,needs_review
0,Problem,Request,0.587,low,high,0.448,True
1,Request,Request,0.797,high,medium,0.419,False
2,Problem,Problem,0.676,medium,medium,0.391,True
3,Change,Change,0.992,low,low,0.421,False
4,Problem,Problem,0.538,low,low,0.777,True
...,...,...,...,...,...,...,...
95,Problem,Incident,0.706,low,medium,0.630,False
96,Problem,Incident,0.546,medium,medium,0.405,True
97,Problem,Incident,0.707,medium,medium,0.385,False
98,Problem,Incident,0.647,low,low,0.438,True


flagged: 58 of 100
(302, 8)
